# Qdrant Vector Database Demo

Qdrant is a purpose-built vector database with excellent filtering and multi-tenancy support.

**Features covered:**
- In-memory setup (no Docker needed)
- CRUD operations
- Filtered search
- Multi-tenancy with payload filtering

**Prerequisites:**
```bash
pip install qdrant-client sentence-transformers
```

In [24]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer
import uuid

# In-memory Qdrant (no Docker needed)
client = QdrantClient(":memory:")

# Load embedding model
encoder = SentenceTransformer("all-MiniLM-L6-v2")
VECTOR_DIM = 384

# Create collection
client.create_collection(
    collection_name="documents",
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE)
)
print("✓ Collection created")

✓ Collection created


---

## 1. Insert Documents with Metadata

In [25]:
# Sample documents with tenant IDs (multi-tenancy)
documents = [
    {"text": "Remote work is allowed 3 days per week", "tenant_id": "acme", "category": "hr"},
    {"text": "Vacation policy: 25 days annual leave", "tenant_id": "acme", "category": "hr"},
    {"text": "Code reviews are mandatory before merge", "tenant_id": "acme", "category": "engineering"},
    {"text": "Work from home requires manager approval", "tenant_id": "globex", "category": "hr"},
    {"text": "All employees get 20 days paid leave", "tenant_id": "globex", "category": "hr"},
]

# Insert documents
points = []
for i, doc in enumerate(documents):
    embedding = encoder.encode(doc["text"]).tolist()
    points.append(PointStruct(
        id=i,
        vector=embedding,
        payload={
            "text": doc["text"],
            "tenant_id": doc["tenant_id"],
            "category": doc["category"]
        }
    ))

client.upsert(collection_name="documents", points=points)
print(f"✓ Inserted {len(points)} documents")

✓ Inserted 5 documents


---

## 2. Basic Search

In [26]:
query = "How many vacation days do I get?"
query_vector = encoder.encode(query).tolist()

# Note: search() was removed in qdrant-client 1.16.0, use query_points() instead
results = client.query_points(
    collection_name="documents",
    query=query_vector,
    limit=3
)

print(f"Search: \"{query}\"")
print("=" * 60)
for r in results.points:
    print(f"  [{r.score:.3f}] {r.payload['text']} (tenant: {r.payload['tenant_id']})")

Search: "How many vacation days do I get?"
  [0.657] Vacation policy: 25 days annual leave (tenant: acme)
  [0.452] All employees get 20 days paid leave (tenant: globex)
  [0.367] Remote work is allowed 3 days per week (tenant: acme)


---

## 3. Filtered Search (Multi-Tenancy)

In [27]:
# Search with tenant filter (crucial for multi-tenancy)
results = client.query_points(
    collection_name="documents",
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="tenant_id", match=MatchValue(value="acme"))]
    ),
    limit=3
)

print(f"Filtered Search (tenant=acme): \"{query}\"")
print("=" * 60)
for r in results.points:
    print(f"  [{r.score:.3f}] {r.payload['text']}")

Filtered Search (tenant=acme): "How many vacation days do I get?"
  [0.657] Vacation policy: 25 days annual leave
  [0.367] Remote work is allowed 3 days per week
  [-0.067] Code reviews are mandatory before merge


In [28]:
# Combine filters: tenant AND category
results = client.query_points(
    collection_name="documents",
    query=encoder.encode("coding standards").tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(key="tenant_id", match=MatchValue(value="acme")),
            FieldCondition(key="category", match=MatchValue(value="engineering"))
        ]
    ),
    limit=3
)

print(f"Filtered Search (tenant=acme AND category=engineering)")
print("=" * 60)
for r in results.points:
    print(f"  [{r.score:.3f}] {r.payload['text']}")

Filtered Search (tenant=acme AND category=engineering)
  [0.258] Code reviews are mandatory before merge


---

## 4. Production Patterns

```python
# Production Qdrant with Docker:
# docker run -p 6333:6333 qdrant/qdrant

# Connect to persistent instance
client = QdrantClient(host="localhost", port=6333)

# Recommended: Create indexes on filter fields
client.create_payload_index(
    collection_name="documents",
    field_name="tenant_id",
    field_schema="keyword"
)

# Note: qdrant-client 1.16.0+ uses query_points() instead of search()
# results = client.query_points(collection_name="docs", query=vector, limit=10)
```

**Multi-tenancy recommendation:**
- Under 100 tenants: Payload filtering (shown above)
- Over 100 tenants, large data: Collection per tenant
- Compliance requirements: Separate Qdrant instances